# nb_rotina_nexti
Rotinas de alta frequência da Nexti, do Bronze à Gold numa única sessão de notebook. Um só agendamento
(a cada 5 min) roda, em sequência, as rotinas de `ROTINAS` que estão ativas, dentro da janela de horário
e com o intervalo vencido. Para cada rotina:
1. `nb_ingestao_api` busca na API só as tabelas da rotina, desde o último watermark;
2. sem nenhuma alteração na API, a rotina termina aqui (só uma linha SEM_ALTERACAO no log);
3. com alteração, `nb_orquestrador` leva o lote para a Silver e as Golds recalculam só o que mudou.

Incluir uma rotina = uma entrada em `ROTINAS`. Rodar uma rotina sob demanda, fora do horário:
executar o notebook com `rotina = "AUSENCIAS"`.

A trava `Files/rotinas/nb_rotina_nexti.lock` (lakehouse Bronze) impede duas execuções simultâneas; a
seguinte termina sem fazer nada. A trava expira em `trava_minutos`, caso uma execução morra sem
liberá-la. As tabelas destas rotinas ficam fora do ciclo geral (TABELAS_ROTINA no nb_ingestao_api).

In [ ]:
rotina = ""                   # vazio = as rotinas no horário (agendamento); "AUSENCIAS" = só ela, a qualquer hora
modo_gold = "INCREMENTAL"     # COMPLETO recalcula as Golds inteiras; NENHUM pula as Golds (carga de histórico)
max_dias_por_execucao = 45    # janela máxima por tabela (recuperação de histórico)
configuracao_api = '{"NEXTI": {"key_vault": "https://kv-dw-fabric-kmf.vault.azure.net/"}}'
trava_minutos = 30
timeout_etapa_s = 3000

In [ ]:
import os, json, datetime as dt
import pyarrow as pa
import duckdb
from deltalake import DeltaTable, write_deltalake

# Não renomear os lakehouses: o banco do SQL endpoint guarda o nome da criação e a sincronização falha
LH_BRONZE, LH_SILVER, LH_GOLD = "Bronze", "Silver", "Gold"
FUSO_BRT = dt.timezone(dt.timedelta(hours=-3))

_LOCAL = os.environ.get("DW_LOCAL_ROOT")  # usado só em testes fora do Fabric
if _LOCAL:
    def _opts():
        return None
    def caminho(lakehouse, tabela):
        return f"{_LOCAL}/{lakehouse}/{tabela}"
    def sair(valor):
        globals()["_EXIT_VALUE"] = valor
else:
    import notebookutils
    _WS = notebookutils.runtime.context["currentWorkspaceName"]
    def _opts():
        return {"bearer_token": notebookutils.credentials.getToken("storage"),
                "use_fabric_endpoint": "true"}
    def caminho(lakehouse, tabela):
        return f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{lakehouse}.Lakehouse/Tables/{tabela}"
    def sair(valor):
        notebookutils.notebook.exit(valor)

def existe(lakehouse, tabela):
    try:
        DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())
        return True
    except Exception:
        return False

def ler(lakehouse, tabela, filtros=None, colunas=None):
    dt_ = DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())
    kw = {k: v for k, v in (("filters", filtros), ("columns", colunas)) if v}
    return dt_.to_pyarrow_table(**kw)

def gravar(lakehouse, tabela, dados, modo="append"):
    if modo == "append":
        acrescentar_colunas(lakehouse, tabela, dados.schema)
    kw = {"schema_mode": "overwrite"} if modo == "overwrite" else {}
    write_deltalake(caminho(lakehouse, tabela), dados, mode=modo,
                    storage_options=_opts(), **kw)

def acrescentar_colunas(lakehouse, tabela, schema):
    """Coluna nova na origem (ex.: campo novo da API): a tabela é regravada com a coluna, vazia nas linhas
    antigas, antes do append ou do MERGE. Não depende da versão do deltalake suportar evolução de schema."""
    if not existe(lakehouse, tabela):
        return
    atual = tabela_delta(lakehouse, tabela)
    nomes = {f.name for f in atual.schema().fields}
    novas = [c for c in schema if c.name not in nomes]
    if not novas:
        return
    dados = atual.to_pyarrow_table()
    for c in novas:
        dados = dados.append_column(c, pa.nulls(dados.num_rows, type=c.type))
    write_deltalake(caminho(lakehouse, tabela), dados, mode="overwrite", schema_mode="overwrite",
                    storage_options=_opts())
    print(f"{tabela}: colunas novas {[c.name for c in novas]}")

def tabela_delta(lakehouse, tabela):
    return DeltaTable(caminho(lakehouse, tabela), storage_options=_opts())

def para_arrow(rel):
    """Converte resultado DuckDB em pyarrow.Table (compatível com várias versões)."""
    r = rel.fetch_arrow_table() if hasattr(rel, "fetch_arrow_table") else rel.arrow()
    return r.read_all() if hasattr(r, "read_all") else r

def nome_bronze(tabela_destino):
    """ctl_tabelas guarda o nome base (fato_contas_receber_gs); cada camada recebe o seu prefixo."""
    return f"bronze_{tabela_destino}"

def nome_silver(tabela_destino):
    return f"silver_{tabela_destino}"

def agora_utc():
    return dt.datetime.now(dt.timezone.utc)

def watermark_gold(gold, fonte):
    """Último _dt_carga da Silver já levado para esta Gold (tabela ctl_gold na Bronze). As Golds são
    incrementais por esta marca, e não pelo lote: se uma Gold falhar, a próxima execução retoma dali."""
    if not existe(LH_BRONZE, "ctl_gold"):
        return None
    for r in ler(LH_BRONZE, "ctl_gold").to_pylist():
        if r["gold"] == gold and r["fonte"] == fonte:
            return r["ultimo_dt_carga"]
    return None

def gravar_watermark_gold(gold, fonte, valor):
    linha = pa.Table.from_pylist([{"gold": gold, "fonte": fonte, "ultimo_dt_carga": valor,
                                   "dt_atualizacao": agora_utc()}],
                                 schema=pa.schema([("gold", pa.string()), ("fonte", pa.string()),
                                                   ("ultimo_dt_carga", pa.timestamp("us", tz="UTC")),
                                                   ("dt_atualizacao", pa.timestamp("us", tz="UTC"))]))
    if not existe(LH_BRONZE, "ctl_gold"):
        gravar(LH_BRONZE, "ctl_gold", linha, modo="overwrite")
        return
    (tabela_delta(LH_BRONZE, "ctl_gold")
        .merge(linha, predicate="t.gold = s.gold AND t.fonte = s.fonte", source_alias="s", target_alias="t")
        .when_matched_update_all().when_not_matched_insert_all().execute())

def consulta(nome):
    """Texto de uma consulta da pasta `consultas` (no Fabric: lakehouse Gold → Files/consultas/<nome>.sql,
    copiada de implantacao/consultas pela implantação). Nos testes locais, lê de DW_CONSULTAS."""
    if _LOCAL:
        with open(os.path.join(os.environ["DW_CONSULTAS"], f"{nome}.sql"), encoding="utf-8") as arq:
            return arq.read()
    return notebookutils.fs.head(
        f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{LH_GOLD}.Lakehouse/Files/consultas/{nome}.sql", 10_000_000)

def cabecalho(sql):
    """Configuração no topo do .sql, em linhas `-- @chave valor` (ex.: `-- @destino gold_fato_coberturas`)."""
    cfg = {}
    for linha in sql.splitlines():
        linha = linha.strip()
        if linha.startswith("-- @"):
            chave, _, valor = linha[4:].partition(" ")
            cfg[chave] = valor.strip()
    return cfg

def ler_config(nome):
    """Arquivo YAML da pasta config (no Fabric: lakehouse Gold → Files/config/<nome>, copiado de config/ pela
    implantação). Nos testes locais, lê de DW_CONFIG."""
    import yaml
    if _LOCAL:
        with open(os.path.join(os.environ["DW_CONFIG"], nome), encoding="utf-8") as arq:
            return yaml.safe_load(arq)
    return yaml.safe_load(notebookutils.fs.head(
        f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{LH_GOLD}.Lakehouse/Files/config/{nome}", 10_000_000))

In [ ]:
import uuid

# tabelas: endpoints da Nexti (ENDPOINTS_NEXTI no nb_ingestao_api)
# golds: em ordem; "nb_..." = notebook próprio, outro nome = consulta consultas/<nome>.sql executada pelo nb_gold
# intervalo_min: de quanto em quanto tempo roda; janela: horário de Brasília em que roda (inclusive)
ROTINAS = {
    "AUSENCIAS": {
        "tabelas": ["WORKPLACETRANSFERS", "ABSENCES"],
        "golds": ["nb_gold_fato_movimentacao_colaborador", "nb_gold_fato_ausencias"],
        "intervalo_min": 5, "janela": ("06:00", "20:00"), "ativo": True,
    },
    "COBERTURAS": {
        "tabelas": ["REPLACEMENTS"],
        "golds": ["gold_fato_coberturas"],
        "intervalo_min": 5, "janela": ("06:00", "20:00"), "ativo": True,
    },
    "MARCACAO": {
        "tabelas": ["CLOCKINGS"],
        "golds": ["gold_fato_marcacao_ponto"],
        "intervalo_min": 5, "janela": ("06:00", "20:00"), "ativo": False,  # ativar após a carga do histórico
    },
    # cadastros de que as Golds dependem: só sob demanda (o ciclo geral COMPLETO os atualiza às 05:00)
    "CADASTROS": {
        "tabelas": ["PERSONS", "WORKPLACES", "COMPANIES", "USERACCOUNTS"],
        "golds": [],
        "intervalo_min": None, "janela": None, "ativo": False,
    },
}
inicio = agora_utc()
_log = []

def log(lote, rot, tabela_id, etapa, linhas, status, msg=""):
    _log.append({"lote": lote, "ciclo": f"ROTINA_{rot}", "tabela_id": tabela_id, "etapa": etapa,
                 "linhas": int(linhas or 0), "status": status, "mensagem": (msg or "")[:4000],
                 "dt_registro": agora_utc()})

def gravar_log():
    if _log:
        schema = ler(LH_BRONZE, "ctl_log").schema
        gravar(LH_BRONZE, "ctl_log", pa.Table.from_pylist(_log, schema=schema))
        _log.clear()

def no_horario(nome, cfg):
    """Ativa, dentro da janela e com o intervalo vencido desde a última execução."""
    if not cfg["ativo"] or not cfg["intervalo_min"]:
        return False
    agora = dt.datetime.now(FUSO_BRT)
    ini, fim = cfg["janela"]
    if not (ini <= agora.strftime("%H:%M") <= fim):
        return False
    ultima = watermark_gold(f"ROTINA_{nome}", "execucao")
    # 1 min de folga: o agendamento não dispara exatamente no mesmo segundo
    return ultima is None or agora_utc() - ultima >= dt.timedelta(minutes=cfg["intervalo_min"] - 1)

if rotina:
    if rotina not in ROTINAS:
        raise ValueError(f"Rotina desconhecida: {rotina}. Disponíveis: {', '.join(ROTINAS)}")
    escolhidas = [rotina]
else:
    escolhidas = [n for n, c in ROTINAS.items() if no_horario(n, c)]

In [ ]:
if _LOCAL:
    def pegar_trava():
        return True
    def soltar_trava():
        pass
    def rodar(nome, params):
        raise RuntimeError("nb_rotina_nexti só roda no Fabric (chama outros notebooks)")
else:
    TRAVA = f"abfss://{_WS}@onelake.dfs.fabric.microsoft.com/{LH_BRONZE}.Lakehouse/Files/rotinas/nb_rotina_nexti.lock"
    DONO = str(uuid.uuid4())

    def pegar_trava():
        if notebookutils.fs.exists(TRAVA):
            try:
                desde = dt.datetime.fromisoformat(notebookutils.fs.head(TRAVA, 100).split("|")[1])
            except Exception:
                desde = agora_utc()
            if agora_utc() - desde < dt.timedelta(minutes=int(trava_minutos)):
                return False
            print(f"trava vencida desde {desde:%H:%M}: assumindo a rotina")
        notebookutils.fs.put(TRAVA, f"{DONO}|{agora_utc().isoformat()}", True)
        return True

    def soltar_trava():
        try:
            if notebookutils.fs.head(TRAVA, 200).startswith(DONO):
                notebookutils.fs.rm(TRAVA)
        except Exception:
            pass

    def rodar(nome, params):
        return notebookutils.notebook.run(nome, int(timeout_etapa_s), params)

def manutencao(nome, cfg, lote):
    """Uma vez por dia: compacta os arquivos pequenos que a rotina gera a cada 5 min e remove os
    antigos (VACUUM de 7 dias). Falha aqui só vai para o log."""
    alvos = [(LH_BRONZE, "ctl_log"), (LH_BRONZE, "ctl_tabelas"), (LH_BRONZE, "ctl_gold")]
    for t in ler(LH_BRONZE, "ctl_tabelas").to_pylist():
        if t["origem"] == "NEXTI" and t["tabela_origem"] in cfg["tabelas"]:
            alvos += [(LH_BRONZE, nome_bronze(t["tabela_destino"])), (LH_SILVER, nome_silver(t["tabela_destino"]))]
    alvos += [(LH_GOLD, g.replace("nb_", "", 1)) for g in cfg["golds"]]
    for lh, tabela in alvos:
        try:
            if existe(lh, tabela):
                alvo = tabela_delta(lh, tabela)
                alvo.optimize.compact()
                alvo.vacuum(retention_hours=168, enforce_retention_duration=False, dry_run=False)
        except Exception as e:
            log(lote, nome, tabela, "MANUTENCAO", 0, "FALHA", str(e))
    # regras de qualidade do grupo da rotina (config/qualidade.yml), uma vez por dia
    try:
        q = json.loads(rodar("nb_qualidade", {"grupo": nome, "lote": lote}) or "{}")
        status = "SUCESSO" if not q.get("falhas_erro") else "FALHA"
        log(lote, nome, f"QUALIDADE.{nome}", "QUALIDADE", q.get("regras", 0), status,
            json.dumps(q, ensure_ascii=False))
    except Exception as e:
        log(lote, nome, f"QUALIDADE.{nome}", "QUALIDADE", 0, "FALHA", str(e))
    gravar_watermark_gold(f"ROTINA_{nome}", "manutencao", agora_utc())

def hoje_ja(nome, marca):
    ultima = watermark_gold(f"ROTINA_{nome}", marca)
    return ultima is not None and ultima.astimezone(FUSO_BRT).date() == dt.datetime.now(FUSO_BRT).date()

def executar(nome, cfg):
    lote = f"rotina-{nome.lower()}-{uuid.uuid4()}"
    t_ini = agora_utc()
    gravar_watermark_gold(f"ROTINA_{nome}", "execucao", t_ini)
    res_api = rodar("nb_ingestao_api", {
        "lote": lote, "ciclo": "FATOS", "origens": "NEXTI", "tabelas": ",".join(cfg["tabelas"]),
        "configuracao_api": configuracao_api, "max_dias_por_execucao": int(max_dias_por_execucao)})
    itens_api = json.loads(res_api or "[]")
    copiadas = sum(int(float(s.split("|")[2] or 0)) for s in itens_api)
    falhou = any(s.split("|")[1] == "FALHA" for s in itens_api)
    # nada mudou na API e as Golds já viraram o dia: termina sem Silver nem Gold. Os watermarks não
    # avançam; a próxima busca parte do mesmo ponto, o que só alarga um pouco a janela consultada.
    if copiadas == 0 and not falhou and modo_gold != "COMPLETO" and (not cfg["golds"] or hoje_ja(nome, "golds_dia")):
        seg = (agora_utc() - t_ini).total_seconds()
        log(lote, nome, f"ROTINA.{nome}", "ROTINA", 0, "SEM_ALTERACAO", f"nenhuma alteração na API | {seg:.0f}s")
        return f"{nome}: sem alteração ({seg:.0f}s)"
    rodar("nb_orquestrador", {
        "lote": lote, "ciclo": "FATOS", "resultados": "[]", "resultados_api": res_api,
        "status_ciclo": "SUCESSO", "origens": "NEXTI"})
    resumo = [f"{copiadas} linhas da API"]
    for g in (cfg["golds"] if modo_gold != "NENHUM" else []):
        t0 = agora_utc()
        if g.startswith("nb_"):
            r = json.loads(rodar(g, {"lote": lote, "modo": modo_gold}) or "{}")
        else:
            r = json.loads(rodar("nb_gold", {"gold": g, "lote": lote, "modo": modo_gold}) or "{}")
        log(lote, nome, r.get("gold", g), "GOLD", r.get("linhas", 0), "SUCESSO",
            f"{json.dumps(r, ensure_ascii=False)} | {(agora_utc() - t0).total_seconds():.0f}s")
        resumo.append(f"{r.get('gold', g)}: {r.get('linhas', 0)} linhas")
    if cfg["golds"] and modo_gold != "NENHUM":
        gravar_watermark_gold(f"ROTINA_{nome}", "golds_dia", agora_utc())
    if not hoje_ja(nome, "manutencao"):
        manutencao(nome, cfg, lote)
        resumo.append("manutenção diária")
    seg = (agora_utc() - t_ini).total_seconds()
    log(lote, nome, f"ROTINA.{nome}", "ROTINA", copiadas, "SUCESSO", " | ".join(resumo) + f" | {seg:.0f}s")
    return f"{nome}: " + " | ".join(resumo) + f" | {seg:.0f}s"

In [ ]:
resultado = {"rotinas": escolhidas, "status": "SEM_ROTINA_NO_HORARIO" if not escolhidas else "EM_EXECUCAO"}
falhas = []
if escolhidas and not pegar_trava():
    print("nb_rotina_nexti já em execução: nada a fazer")
elif escolhidas:
    try:
        for nome in escolhidas:
            try:
                print(executar(nome, ROTINAS[nome]))
            except Exception as e:
                falhas.append(f"{nome}: {e}")
                log("", nome, f"ROTINA.{nome}", "ROTINA", 0, "FALHA", str(e))
            gravar_log()
        resultado = {"rotinas": escolhidas, "status": "FALHA" if falhas else "SUCESSO",
                     "segundos": round((agora_utc() - inicio).total_seconds())}
    finally:
        soltar_trava()
else:
    print("nenhuma rotina no horário")

In [ ]:
# monitoramento a cada 30 min (rotinas atrasadas, tabelas paradas, falhas, qualidade) -> ctl_alertas
ultimo_monitor = watermark_gold("MONITORAMENTO", "ctl_log")
if not _LOCAL and (ultimo_monitor is None or agora_utc() - ultimo_monitor > dt.timedelta(minutes=30)):
    try:
        print("monitoramento:", rodar("nb_monitoramento", {}))
    except Exception as e:
        print(f"monitoramento falhou: {str(e)[:300]}")

In [ ]:
# falha de uma rotina não impede as outras; no fim, a execução fica marcada como falha no Fabric
if falhas:
    raise RuntimeError("Rotinas com falha:\n" + "\n".join(falhas))
# fora do try: no Fabric o exit pode ser tratado como exceção e cair no except
sair(json.dumps(resultado))